In [ ]:
from abc import ABC,abstractmethod 
import pandas as pd
import mysql.connector
from pathlib import Path
import os

df_config = {'host':'localhost',
             'user':'demo_user',
             'password':'demo_pass',
             'database':'db'}   
#df_config need to be set up before you extract sql file from your sqldatabase 
#do not change the keys but change the values of user and password as per your credentials you can even make env file which makes it much more secure

# this is the main File loading class 
class baseloader(ABC):
    def __init__(self,source_path):
        self._source_path = source_path  
        self.set_string(source_path)

    def set_string(self,value): # validator for the file path 
        
        if isinstance(value,dict):
            if not all (x in value for x in ['host','user','password','database']): # 1st checkpoint for checking all keys present  
                raise KeyError('Key is missing')
            if any(not str(value[k]).strip() for k in ['host','user','password','database']): # 2nd checkpoint to check if values are present 
                raise Exception('Missing Credentials')
            else:
                new_value = value        
        elif isinstance(value,str):
            new_value = value 
        self._source_path = new_value 
    
    @abstractmethod
    def extract(self,):
        pass

# CSV file loader 
class CSVLoader(baseloader):
    def __init__(self,source_path):
        super().__init__(source_path)

    def extract(self): # file extraction processed here
        try:
            path = Path(self._source_path)  # using Path here will also validate the path and raise an error if it is invalid
            df = pd.read_csv(path,nrows=100000)
        except FileNotFoundError:
            print(f'File at {self._source_path} not found')
        except Exception as e:
            print(e)
        else:
            return df

# MYSQL Database loader 
class SQLLoader(baseloader):
    
    def __init__(self,source_path,query):
        super().__init__(source_path)
        self.query = query

    def extract(self): # SQL file extraction to pandas 
        conn = None
        try:    
            conn = mysql.connector.connect(**self._source_path) # **self._source_path here ** is used to unpack dict 
            cursor = conn.cursor()
            cursor.execute(self.query) 
        except Exception as e:
            print(f'[SystemError]{e}')
        else:
            df = pd.DataFrame(cursor.fetchall(),columns=[col[0] for col in cursor.description]) # for coloumns you need to use col[0] as all the columns name are in the 0 index
            return df
        finally:
            if conn and conn.is_connected():
                conn.close()
            
csv = CSVLoader(source_path=r'C:\Users\rawat\Downloads\superstore.csv')
# sql = SQLLoader(source_path= db_config,query='select * from products')

# Extracted files by both classes 
# raw_data_sql = sql.extract() 
raw_data_csv = csv.extract()

# Data will be processed here 
class datatransformers:
    
    # Data column normalized to lower and spaces removed 
    def clean_columns_name(self,df:pd.DataFrame):
        temp_df = df.copy()
        temp_df.columns = temp_df.columns.str.lower().str.replace('., ','_').str.strip()
        return temp_df
    
    # To check for any missing values 
    def data_integrity(self,df=pd.DataFrame):
        temp_df = df.copy()
        missing_info = temp_df.isnull().any()
        return missing_info

    # Data Imputation process 
    def data_imputation(self, df: pd.DataFrame, method=None, fill_value='Unknown', columns=None):
        temp_df = df.copy()
    
        # which columns to process
        # if columns is None, use all columns; otherwise, use the provided list
        target_cols = columns if columns is not None else temp_df.columns
        
        # 2. Apply the chosen method only to those specific columns
        if method == 'ffill':
            temp_df[target_cols] = temp_df[target_cols].ffill()
        elif method == 'bfill':
            temp_df[target_cols] = temp_df[target_cols].bfill()
        elif method == 'drop na rows':
            # This will only remove rows if the target columns in the subset are null use carefully 
            temp_df = temp_df.dropna(subset=target_cols)
        elif method == 'fillna' or method is None:
            temp_df[target_cols] = temp_df[target_cols].fillna(fill_value)
        elif method == 'interpolate':
            # Remove inplace=True so it returns the data back to the columns
            temp_df[target_cols] = temp_df[target_cols].interpolate(method='linear')
        return temp_df

cleaning = datatransformers() # creating a data cleaning instance
cleaned = cleaning.clean_columns_name(raw_data_csv) # passing the loaded data into the cleaning process 

from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError
from urllib.parse import quote_plus

class DatabaseWriter:
    def __init__(self, config):
        database = config.get('database')
        # Encode the password to handle special characters like '@'
        safe_password = quote_plus(config['password'])
    
        self.db_url = (
            f"mysql+mysqlconnector://{config['user']}:{safe_password}"
            f"@{config['host']}/{database}"
        )
        self.engine = create_engine(self.db_url)

    def load_data(self, df, table_name):
        try:
            if df is not None and not df.empty:
                # index=False prevents pandas from adding an extra column for row numbers
                df.to_sql(name=table_name, con=self.engine, if_exists='replace', index=False)
                print(f"Successfully loaded to {table_name}")
            else:
                print("No data found to load.")
        except SQLAlchemyError as e:
            print(f"Database Load Error: {e}")

# db = DatabaseWriter(db_config)
# db.load_data(cleaned,'database')